In [33]:
import functools
import os
import sys
import json

from ultralytics import YOLO
import pandas as pd

sys.path.append("..")
sys.path.append("../..")
import ultralytics_validator_wrapper
from tflite_inference import TFLiteInferer, TFLiteMerged
from tflite_inference_input_adapter import TFLiteInferenceWithInputAdapter

pd.options.display.float_format = '{:.3f}'.format

%matplotlib inline
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [34]:
BASE_EXPERIMENT_DIR = "../results/cam3_revB_batch1_7_margin_15_yolov8n_bg_wb_alif_clahe_box_01_dfl_01_2nd"
BASE_WEIGHTS_DIR = os.path.join(BASE_EXPERIMENT_DIR, "weights")
WEIGHTS_PTH = os.path.join(BASE_WEIGHTS_DIR, "best.pt")

HOLDOUT_PTH = "/home/klymy/data/cam4_revB/yolo_dataset/batch1_5_margin_15_bg_wb_alif_clahe/project_holdout.yaml"
VAL_PTH = "/home/klymy/data/cam4_revB/yolo_dataset/batch1_7_margin_15_bg_wb_alif_clahe/project.yaml"
POC_PTH = "/home/klymy/projects/researches/people_detection_pg2/quality_reduce_comparison/cam4_poc_15/project_pg.yaml"

if os.path.exists(os.path.join(BASE_EXPERIMENT_DIR, "inference_params.json")):
    with open(os.path.join(BASE_EXPERIMENT_DIR, "inference_params.json"), "r") as f:
        data = json.load(f)
    CONF_TH = float(data["conf_th"])
    IOU_TH = float(data["iou_th"])
else:
    print("No inference_params.json found, using default thresholds.")
    CONF_TH = 0.35
    IOU_TH = 0.50
    
IOU_TH = 0.35

args = dict(
    task="pose",
    device=[0],
    batch=128,
    verbose=False,
    conf=CONF_TH,
    iou=IOU_TH,
    imgsz=320,
    model="val",
    rect=False,
    single_cls=False,
)

In [35]:
model = YOLO(WEIGHTS_PTH)

In [36]:
inferer = TFLiteMerged(
    yolo_pth=os.path.join(BASE_WEIGHTS_DIR, "best_non_separated_saved_model/best_full_integer_quant.tflite"),
    conf_th=CONF_TH,
    iou_th=IOU_TH,
    imgsz=320,
)

# inferer_adapter = TFLiteInferenceWithInputAdapter(
#     yolo_pth=os.path.join(BASE_WEIGHTS_DIR, "best_saved_model/best_full_integer_quant.tflite"),
#     head_pth=os.path.join(BASE_WEIGHTS_DIR, "head_saved_model/head_float32.tflite"),
#     adapter_pth="../input_adapter/input_adapter_full_integer_quant.tflite",
#     conf_th=CONF_TH,
#     iou_th=IOU_TH,
#     imgsz=320,
# )

In [37]:
args["data"] = VAL_PTH

ultralytics_validator_wrapper.validate_model(
    args,
    model,
    functools.partial(ultralytics_validator_wrapper.predict_tflite, inferer, args),
    False,
)

val: Scanning /home/klymy/data/cam4_revB/yolo_dataset/batch1_7_margin_15_bg_wb_alif_clahe/labels/val.cache... 470 images, 1 backgrounds, 0 corrupt: 100%|██████████| 470/470 [00:00<?, ?it/s]


tst validation: 100%|██████████| 4/4 [00:19<00:00,  4.81s/it]


                   all        470       2347      0.809      0.895      0.822      0.261      0.918      0.953      0.973      0.972
   task    F1     P     R  MAP@50
0   box 0.849 0.809 0.895   0.822
1  pose 0.935 0.918 0.953   0.973


In [38]:
args["data"] = HOLDOUT_PTH

ultralytics_validator_wrapper.validate_model(
    args,
    model,
    functools.partial(ultralytics_validator_wrapper.predict_tflite, inferer, args),
    False,
)


val: Scanning /home/klymy/data/cam4_revB/yolo_dataset/batch1_5_margin_15_bg_wb_alif_clahe/labels/holdout.cache... 148 images, 5 backgrounds, 0 corrupt: 100%|██████████| 148/148 [00:00<?, ?it/s]


tst validation: 100%|██████████| 2/2 [00:06<00:00,  3.16s/it]


                   all        148        409      0.726        0.7      0.671      0.214      0.911      0.879      0.935      0.927
   task    F1     P     R  MAP@50
0   box 0.713 0.726 0.700   0.671
1  pose 0.895 0.911 0.879   0.935


In [39]:
args["data"] = POC_PTH

ultralytics_validator_wrapper.validate_model(
    args,
    model,
    functools.partial(ultralytics_validator_wrapper.predict_tflite, inferer, args),
    False,
)


val: Scanning /home/klymy/projects/researches/people_detection_pg2/quality_reduce_comparison/cam4_poc_15/labels/val.cache... 191 images, 5 backgrounds, 0 corrupt: 100%|██████████| 191/191 [00:00<?, ?it/s]


tst validation: 100%|██████████| 2/2 [00:07<00:00,  3.99s/it]


                   all        191        417      0.681      0.602      0.654       0.19      0.707      0.789      0.795      0.781
   task    F1     P     R  MAP@50
0   box 0.639 0.681 0.602   0.654
1  pose 0.746 0.707 0.789   0.795
